# 24 AG stage 2: the ledger on every policy run

**Purpose.** To see how the stage-1 winners and losers change across cost samples, market conditions,
the emissions constraint and the credit designs, this notebook applies the stage-1 ledger code
(`aglib/ledger.py`, unchanged) to every stage-2 run in protocol §2 (signed 2026-10-28; amendments A1–A15;
stage-2b baselines per amendment A22, drafted 2026-11-16 from Ethan's choice on review page item 9).

| Stage | Policy runs | Baseline | Difference measured |
|---|---|---|---|
| 2a | the other 60 of the 66 anchors (P05/P95, P25/P75, 36 Monte Carlo draws) | `fbR_none` | programme |
| 2b | 178 runs in six market conditions | primary: `nomand_fbR_none_{cond}` (batch `ag`); also `fbR_none` | programme, in the condition (primary); condition + programme (against `fbR_none`) |
| 2c | 30 runs under the national emissions cap | each of the 8 cap baselines; primary `bx_aj_p75_f50_co2cap` | programme, under the cap |
| 2d | 40 credit runs (`fb{A,B,C,K}`, `fbBm*`, `fbCm*`) | `fbR_none` | credit design (actual tax expenditure; no funding cases) |

**What it writes** (`results/exports/24/`): `ag2_pairs.csv`, `ag2_actor_pv.parquet`,
`ag2_views.parquet`, `ag2_closure.parquet`, `ag2_c3.csv`, `ag2_allowance_checks.csv`. It opens no ReEDS
outputs file and states no hypothesis. Observations go to `docs/observations.md` by hand (protocol §10).


In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import re
import time

import numpy as np
import pandas as pd

import nlpaper
from nlpaper import paths
from aglib import ledger as L
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 30)
OUT = paths.exports("24")
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
FLAG = 0.005
EXT = L.load_external()
CRF = L.crf_after_2026()

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


## 1. The pairs (protocol §2)

The run sets are selected from the stage-2 derived extracts with the same rules as notebook 20 §4.

In [2]:
T6 = "(eia|aj|iaea|mck|cop28|eo)"
COND = "(gaslo|gashi|demhi|relo|rehi|translim)"
RULES = {
    "2a": [("smr100", rf"smr100_{T6}_p(05|95)"), ("ratedesign", rf"smr100_{T6}_p(25|75)"),
           ("ratedesign2", rf"ma_{T6}_d\d+")],
    "2b": [("step4", rf"smr100_{T6}_p(05|50|95)_{COND}"), ("batch3", rf"smr100_{T6}_p(25|75)_{COND}")],
    "2c": [("batch3", rf"smr100_{T6}_p\d\d_co2cap")],
    "2d": [("itcfb", r"fb[ABCK]_.+"), ("itcfbm", r"fb[BC]m\d\d_.+")],
}
REF_BASE = ("itcfb", "fbR_none")
COND_BASE = lambda cond: ("ag", f"nomand_fbR_none_{cond}")    # amendment A22: one no-mandate run per condition
CAP_BASES = [("batch3", f"bx_{x}_f50_co2cap") for x in ["aj_p75", "cop28_p50", "cop28_p75", "eia_p75", "eo_p75",
                                                        "iaea_p50", "iaea_p75", "mck_p75"]]
CAP_PRIMARY = ("batch3", "bx_aj_p75_f50_co2cap")
reduced = lambda b: set(pd.read_parquet(L.EXTRACTS_AG / b / "derived_ledger.parquet", columns=["case"])["case"])
rows = []
for st, rules in RULES.items():
    for b, rx in rules:
        for c in sorted(x for x in reduced(b) if re.fullmatch(rx, x)):
            tgt = re.search(T6, c).group(1)
            cond = (re.search(COND + "$", c).group(1) if st == "2b" else ("co2cap" if st == "2c" else "reference"))
            bases = CAP_BASES if st == "2c" else ([COND_BASE(cond), REF_BASE] if st == "2b" else [REF_BASE])
            for bb in bases:
                rows.append(dict(stage=st, batch=b, case=c, target=tgt, condition=cond, base_batch=bb[0],
                                 baseline=bb[1],
                                 primary=(bb == CAP_PRIMARY) if st == "2c" else ((bb != REF_BASE) if st == "2b" else True),
                                 mandate=(st != "2d"),
                                 difference=("condition + programme" if (st == "2b" and bb == REF_BASE) else
                                             dict(a="programme", b="programme, in the condition",
                                                  c="programme, under the cap", d="credit design")[st[1]])))
PAIRS = pd.DataFrame(rows)
PAIRS.to_csv(OUT / "ag2_pairs.csv", index=False)
print(PAIRS.groupby("stage").agg(runs=("case", "nunique"), pairs=("case", "size")))
assert PAIRS.groupby("stage")["case"].nunique().to_dict() == {"2a": 60, "2b": 178, "2c": 30, "2d": 40}

       runs  pairs
stage             
2a       60     60
2b      178    356
2c       30    240
2d       40     40


In [3]:
# Stage-2b condition baselines: verification (protocol §7 step 1, as notebook 22 applies it; amendment A22)
gdb = pd.read_csv(paths.REPO / "inputs" / "capacity_exogenous" / "ReEDS_generator_database_final_EIA-NEMS.csv",
                  usecols=["tech", "StartYear", "summer_power_capacity_MW"], low_memory=False)
EXO_MW = float(gdb[(gdb["tech"] == "nuclear") & (gdb["StartYear"] > 2023)]["summer_power_capacity_MW"].sum())
rd = lambda k: pd.read_parquet(L.EXTRACTS_AG / "ag" / f"{k}.parquet")
cn, cap, dl, pvf = rd("cap_new_ivrt"), rd("cap_nat"), rd("dropped_load"), rd("pvf_onm")
rows = []
for c in sorted(PAIRS.loc[PAIRS["base_batch"] == "ag", "baseline"].unique()):
    a = cn[(cn["case"] == c) & (cn["t"].astype(int) >= 2026)]
    s = cap[(cap["case"] == c) & (cap["i"] == "nuclear-smr")]
    d = dl[(dl["case"] == c) & (dl["t"].astype(int) >= 2026)]
    rows.append(dict(case=c, last_year=int(pvf.loc[pvf["case"] == c, "t"].astype(int).max()),
                     smr_new_MW=float(a.loc[a["i"] == "nuclear-smr", "Value"].sum()),
                     smr_fleet_MW_max=float(s["Value"].max()) if len(s) else 0.0,
                     large_new_MW=float(a.loc[a["i"] == "nuclear", "Value"].sum()), exogenous_MW=EXO_MW,
                     dropped_load_MWh=float(d["Value"].abs().sum()) if len(d) else 0.0))
VCHK = pd.DataFrame(rows)
VCHK["verified"] = ((VCHK["smr_new_MW"] < 1e-6) & (VCHK["smr_fleet_MW_max"] < 1e-6)
                    & ((VCHK["large_new_MW"] - VCHK["exogenous_MW"]).abs() < 0.5)
                    & (VCHK["dropped_load_MWh"] < 1e-6) & (VCHK["last_year"] == 2050))
VCHK.to_csv(OUT / "ag2_condition_baseline_check.csv", index=False)
print(VCHK.round(1).to_string(index=False))
assert len(VCHK) == 6 and VCHK["verified"].all()


                    case  last_year  smr_new_MW  smr_fleet_MW_max  large_new_MW  exogenous_MW  dropped_load_MWh  verified
   nomand_fbR_none_demhi       2050         0.0               0.0        3286.7        3286.7               0.0      True
   nomand_fbR_none_gashi       2050         0.0               0.0        3286.7        3286.7               0.0      True
   nomand_fbR_none_gaslo       2050         0.0               0.0        3286.7        3286.7               0.0      True
    nomand_fbR_none_rehi       2050         0.0               0.0        3286.7        3286.7               0.0      True
    nomand_fbR_none_relo       2050         0.0               0.0        3286.7        3286.7               0.0      True
nomand_fbR_none_translim       2050         0.0               0.0        3286.7        3286.7               0.0      True


## 2. The ledger on every pair

For each pair: the actor PVs, both resident views at every variant, the market checks (C1, C2 and the
RPS account per run-year) and the C3 decomposition into named terms (as notebook 21 §4).

In [4]:
AP, VW, CK, C3, AC = [], [], [], [], []
t0 = time.time()
for k, r in PAIRS.iterrows():
    tag = dict(stage=r["stage"], case=r["case"], target=r["target"], condition=r["condition"],
               baseline=r["baseline"], primary=r["primary"])
    o = L.pair((r["batch"], r["case"]), (r["base_batch"], r["baseline"]), EXT, CRF)
    AP.append(L.actor_pvs(o, r["mandate"]).assign(**tag))
    v = L.views(o, r["mandate"])
    VW.append(v.assign(**tag))
    ck = o["checks"].assign(**tag)
    CK.append(ck)
    c = ck.set_index(["who", "t"])
    d = lambda col: c.loc["run", col] - c.loc["base", col]
    named = (d("C1_residual") - d("C1_vsc_rent")) + (d("C2_residual") - d("C2_vsc_rent")) + d("RPS_residual") - d("RPS_assigned")
    for (h, rk), g in v.groupby(["horizon", "rate"]):
        C3.append(dict(**tag, horizon=h, rate=rk, C3=g["C3_residual"].iloc[0],
                       C3_spread=g["C3_residual"].max() - g["C3_residual"].min(),
                       unexplained=g["C3_residual"].iloc[0] - L.pv(L.flow(named, h), h, L.RATES[rk])))
    m = o["memo_run"]
    AC.append(dict(**tag, emit_irt_minus_emit_nat_Mt=float((m["co2_process_irt"] - m["co2_process_nat"]).abs().max() / 1e6),
                   co2_price_max=float(m["co2_price"].max()),
                   state_import_part_B_min=float(m["state_import_part"].min() / 1e9),
                   rggi_slack_B_maxabs=float(m["rggi_slack"].abs().max() / 1e9)))
    if (k + 1) % 50 == 0:
        print(f"{k + 1} of {len(PAIRS)} pairs, {(time.time() - t0) / 60:.1f} min", flush=True)
AP = pd.concat(AP, ignore_index=True); VW = pd.concat(VW, ignore_index=True)
CK = pd.concat(CK, ignore_index=True); C3 = pd.DataFrame(C3); AC = pd.DataFrame(AC)
AP.to_parquet(OUT / "ag2_actor_pv.parquet", compression="zstd", index=False)
VW.to_parquet(OUT / "ag2_views.parquet", compression="zstd", index=False)
CK.to_parquet(OUT / "ag2_closure.parquet", compression="zstd", index=False)
C3.to_csv(OUT / "ag2_c3.csv", index=False)
AC.to_csv(OUT / "ag2_allowance_checks.csv", index=False)
print(f"{len(PAIRS)} pairs in {(time.time() - t0) / 60:.1f} min;", {k: len(v) for k, v in dict(AP=AP, VW=VW, CK=CK).items()})

50 of 696 pairs, 2.6 min


100 of 696 pairs, 5.2 min


150 of 696 pairs, 7.7 min


200 of 696 pairs, 10.3 min


250 of 696 pairs, 12.7 min


300 of 696 pairs, 15.3 min


... (4 outputs left out)

550 of 696 pairs, 37.2 min


600 of 696 pairs, 43.2 min


650 of 696 pairs, 49.4 min


696 pairs in 51.2 min; {'AP': 3114288, 'VW': 5870880, 'CK': 16704}


## 3. Closure checks

C1 and C2 residuals per run-year as a share of gross payments, flagged above 0.5% (protocol §6); the RPS
account stays open by construction (named residual, amendment A12). C3 must equal the change in the
named market residuals: the largest unexplained amount and the spread of C3 across variants.

In [5]:
x = CK[CK["who"] == "run"].copy()
for c in ("C1", "C2"):
    x[f"{c}_share"] = (x[f"{c}_residual"] - x[f"{c}_vsc_rent"]).abs() / x[f"{c}_gross"].abs()
print("share of gross, max by stage:")
print(x.groupby("stage")[["C1_share", "C2_share"]].max().to_string())
print("run-years flagged (> 0.5% of gross):")
print(x.assign(C1_flag=x["C1_share"] > FLAG, C2_flag=x["C2_share"] > FLAG).groupby("stage")[["C1_flag", "C2_flag"]].sum())
print("\nC3: max |unexplained| and max spread across variants (2025 $B), by stage:")
print(C3.groupby("stage")[["unexplained", "C3_spread"]].agg(lambda s: s.abs().max()).to_string())
print("\nallowance checks, by stage:")
print(AC.groupby("stage")[["emit_irt_minus_emit_nat_Mt", "co2_price_max", "state_import_part_B_min", "rggi_slack_B_maxabs"]]
      .agg(["min", "max"]).round(4).T.to_string())

share of gross, max by stage:
       C1_share  C2_share
stage                    
2a     0.000245  0.000117
2b     0.000374  0.000190
2c     0.000290  0.000119
2d     0.000240  0.000115
run-years flagged (> 0.5% of gross):
       C1_flag  C2_flag
stage                  
2a           0        0
2b           0        0
2c           0        0
2d           0        0

C3: max |unexplained| and max spread across variants (2025 $B), by stage:
       unexplained  C3_spread
stage                        
2a        0.000669   0.000005
2b        0.000972   0.000006
2c        0.000949   0.000071
2d        0.000456   0.000005

allowance checks, by stage:
stage                               2a      2b        2c      2d
emit_irt_minus_emit_nat_Mt min  0.0000  0.0000    0.0000  0.0000
                           max  0.0001  0.0001    0.0001  0.0001
co2_price_max              min  0.0000  0.0000  119.9863  0.0000
                           max  0.0000  0.0000  152.6201  0.0000
state_import_part_B_min 

## 4. Actor totals by stage

Reference display as in notebook 21 §6: horizon A, 2.0%, lump support, funding T (mandate runs) or none
(credit runs), allowances free. Each cell is the median across the runs of that cost target, with the
range in the second table. Under the cap, the primary cap baseline only. 2025 $B, PV to 2026; a positive
value is a gain.

In [6]:
FREE = lambda d: d[d["allowance_state"].isin(["free", "none"]) & d["allowance_national"].isin(["free", "none"])]
sel = FREE(AP[(AP["horizon"] == "A") & AP["support_form"].isin(["lump", "none"]) & AP["funding"].isin(["T", "none"])
              & AP["primary"]])
money = sel[sel["actor"] != "environment"].groupby(["stage", "condition", "case", "target", "actor"])["PV_2.0%"].sum()
for st in ("2a", "2c", "2d"):
    m = money.xs(st, level="stage").groupby(["actor", "target"])
    tab = m.median().unstack("target").reindex(columns=TARGETS)
    rng = (m.min().round(1).astype(str) + " to " + m.max().round(1).astype(str)).unstack("target").reindex(columns=TARGETS)
    n = money.xs(st, level="stage").reset_index().groupby("target")["case"].nunique().reindex(TARGETS)
    print(f"\nstage {st}: median across runs (runs per target: {n.to_dict()})")
    print(tab[(tab.abs() > 0.05).any(axis=1)].round(1).to_string())
    print(f"stage {st}: range")
    print(rng.loc[tab[(tab.abs() > 0.05).any(axis=1)].index].to_string())


stage 2a: median across runs (runs per target: {'eia': 10, 'aj': 10, 'iaea': 10, 'mck': 10, 'cop28': 10, 'eo': 10})
target                              eia    aj   iaea    mck  cop28     eo
actor                                                                    
electricity consumers              34.8  56.8  127.6  163.8  348.9  582.0
foreign buyers                      7.6   7.7   21.6   27.9   61.9   90.4
foreign sellers: Canadian imports  -1.2  -1.7   -3.2   -3.4   -5.0   -6.3
gas producers                     -26.2 -26.2  -74.7  -95.2 -206.2 -294.8
network                            -0.5  -2.4   -4.0   -6.0  -11.5  -16.8
non-power gas consumers            10.4  10.4   29.4   37.8   84.1  122.7
owners: H2 and DAC production      -0.0  -0.3   -0.2   -0.3   -0.2   -0.3
owners: SMR (new)                   8.9  10.2   17.8   17.3   16.4   -5.9
owners: coal                       -0.6  -0.2   -2.4   -2.8   -6.5  -11.7
owners: existing nuclear           -4.2  -5.9  -15.2  -19.4  -42.8  -

**Stage 2b, by market condition.** First table: the programme in each condition, against that condition's
no-mandate run (primary, A22). Second table: condition + programme, against `fbR_none`. Third table: the
condition alone, `nomand_fbR_none_{cond}` − `fbR_none` (one baseline pair, the same for every target).
Median across the runs of each cost target, for the main actors.

In [7]:
b = money.xs("2b", level="stage").reset_index()
main = ["electricity consumers", "taxpayers", "gas producers", "non-power gas consumers", "owners: gas",
        "owners: wind + solar", "owners: existing nuclear", "owners: SMR (new)", "network"]
t = b[b["actor"].isin(main)].groupby(["actor", "condition", "target"])["PV_2.0%"].median().unstack("target")[TARGETS]
print("programme, in the condition (primary):")
print(t.round(1).to_string())
mix = FREE(AP[(AP["stage"] == "2b") & ~AP["primary"] & (AP["horizon"] == "A") & (AP["support_form"] == "lump")
              & (AP["funding"] == "T") & (AP["actor"] != "environment")])
mix = mix.groupby(["condition", "case", "target", "actor"])["PV_2.0%"].sum().reset_index()
tm = mix[mix["actor"].isin(main)].groupby(["actor", "condition", "target"])["PV_2.0%"].median().unstack("target")[TARGETS]
print("\ncondition + programme, against fbR_none:")
print(tm.round(1).to_string())
print("\ncondition alone (the difference of the two tables, per run, then the median):")
cmp = (mix.set_index(["condition", "case", "target", "actor"])["PV_2.0%"]
       - b.set_index(["condition", "case", "target", "actor"])["PV_2.0%"]).dropna().reset_index()
tc = cmp[cmp["actor"].isin(main)].groupby(["actor", "condition", "target"])["PV_2.0%"].median().unstack("target")[TARGETS]
print(tc.round(1).to_string())

programme, in the condition (primary):
target                               eia    aj   iaea    mck  cop28     eo
actor                    condition                                        
electricity consumers    demhi      29.6  44.3  130.0  154.1  305.2  504.7
                         gashi      30.3  35.9  123.5  164.0  360.7  696.4
                         gaslo      38.3  52.9  189.4  229.3  478.0  771.3
                         rehi       43.6  51.0  160.0  194.3  395.6  648.8
                         relo       23.2  37.7  118.8  142.5  289.2  501.4
                         translim   58.8  79.9  184.0  214.1  391.8  646.8
gas producers            demhi     -19.2 -32.1  -78.5  -91.5 -190.4 -299.1
                         gashi      -2.0  -2.1   -8.5   -9.4  -16.8  -30.6
                         gaslo     -32.9 -20.2 -122.6 -150.7 -317.3 -507.2
                         rehi      -36.5 -40.0 -121.3 -141.0 -290.4 -450.2
                         relo      -16.7 -17.3  -56.8  -64.5 


condition + programme, against fbR_none:
target                                 eia      aj    iaea     mck   cop28      eo
actor                    condition                                                
electricity consumers    demhi     -1593.8 -1579.1 -1493.4 -1469.3 -1318.2 -1118.8
                         gashi     -1059.3 -1053.7  -966.2  -925.6  -728.9  -393.3
                         gaslo       784.3   798.8   935.4   975.2  1224.0  1517.2
                         rehi       -689.0  -681.6  -572.5  -538.3  -337.0   -83.8
                         relo        676.4   690.8   772.0   795.7   942.4  1154.6
                         translim   -211.7  -190.5   -86.5   -56.4   121.3   376.3
gas producers            demhi       222.6   209.7   163.4   150.3    51.4   -57.2
                         gashi      2507.9  2507.8  2501.4  2500.5  2493.0  2479.3
                         gaslo      -737.6  -724.9  -827.3  -855.4 -1022.0 -1211.9
                         rehi        289.4   

## 5. Resident views by stage

Reference view variant (AP2 ACS, MCPF 1, φ = 0, allowances free), global EPA 2023 2.0% and domestic
EPA 2019 3%, horizon A: median and range across the runs of each cost target.

In [8]:
ref = FREE(VW[(VW["horizon"] == "A") & VW["support_form"].isin(["lump", "none"]) & VW["funding"].isin(["T", "none"])
             & (VW["health_model"] == "AP2 ACS") & (VW["mcpf"] == 1.0) & (VW["phi"] == 0.0) & VW["primary"]
             & VW["scc_basis"].isin(["global EPA2023 2.0%", "domestic EPA2019 3%"])])
g = ref.groupby(["scc_basis", "stage", "condition", "target"])
out = pd.DataFrame({"runs": g.size(), "view1_median": g["view1"].median(), "view1_min": g["view1"].min(),
                    "view1_max": g["view1"].max(), "view2_median": g["view2"].median(),
                    "view2_min": g["view2"].min(), "view2_max": g["view2"].max()})
print(out.round(1).to_string())

                                            runs  view1_median  view1_min  view1_max  view2_median  view2_min  view2_max
scc_basis           stage condition target                                                                              
domestic EPA2019 3% 2a    reference aj        10          34.5      -17.0       61.1         -25.2      -73.6       -6.6
                                    cop28     10         169.8     -185.9      324.8        -216.1     -573.6      -49.8
                                    eia       10           7.8      -44.0       33.8         -34.3      -81.2       -8.9
                                    eo        10         185.6     -299.2      552.1        -441.4     -927.1      -51.6
                                    iaea      10          58.6     -111.6      252.6         -78.0     -242.0       -0.2
                                    mck       10          28.3     -106.0      150.3        -159.1     -285.1      -26.4
                    2b    demhi 

## 6. Stage 2c: allowance treatments and the cap-baseline band

Under the national cap, `allowance_national` and `allowance_state` each take `free` or `auctioned`
(amendments A14–A15). The first table gives View 1 and View 2 per treatment for the primary cap baseline
(median across the five cost samples of each target). The second gives the band of View 1 and View 2
across the 8 cap baselines, allowances free.

In [9]:
c = VW[(VW["stage"] == "2c") & (VW["horizon"] == "A") & (VW["support_form"] == "lump") & (VW["funding"] == "T")
       & (VW["health_model"] == "AP2 ACS") & (VW["mcpf"] == 1.0) & (VW["phi"] == 0.0)
       & (VW["scc_basis"] == "global EPA2023 2.0%")]
p = c[c["primary"]].groupby(["allowance_national", "allowance_state", "target"])[["view1", "view2"]].median()
print(p.unstack("target").round(1).to_string())
f = FREE(c)
band = f.groupby(["target", "case"]).agg(view1_min=("view1", "min"), view1_max=("view1", "max"),
                                         view2_min=("view2", "min"), view2_max=("view2", "max"))
band["view1_width"] = band["view1_max"] - band["view1_min"]
band["view2_width"] = band["view2_max"] - band["view2_min"]
print("\nwidth across the 8 cap baselines (2025 $B), max per target:")
print(band.groupby("target")[["view1_width", "view2_width"]].max().reindex(TARGETS).round(2).to_string())

                                   view1                                view2                             
target                                aj  cop28  eia     eo  iaea   mck    aj cop28   eia   eo  iaea   mck
allowance_national allowance_state                                                                        
auctioned          auctioned         0.8  203.6 -2.4  420.1  45.6  63.9 -15.6 -50.7 -20.7  5.8 -51.5 -35.1
                   free              1.1  205.5 -2.1  418.5  48.0  65.0 -15.6 -50.7 -20.7  5.8 -51.5 -35.1
                   policy            1.2  206.1 -1.9  421.2  48.4  65.5 -15.6 -50.7 -20.7  5.8 -51.5 -35.1
free               auctioned         7.2  258.0  5.1  520.1  66.3  93.5 -15.6 -50.7 -20.7  5.8 -51.5 -35.1
                   free              7.5  259.9  5.4  518.5  68.7  94.7 -15.6 -50.7 -20.7  5.8 -51.5 -35.1
                   policy            7.6  260.6  5.6  521.3  69.1  95.2 -15.6 -50.7 -20.7  5.8 -51.5 -35.1

width across the 8 cap baselines (20

## 7. What reads these exports

The H3 review (Ethan and Wesley) reads the stage tables at contribution level. Notebook 23 (markets and
technologies) uses the same pairs.